In [2]:
import sys

sys.path.append("../src")

from dotenv import load_dotenv
import os



In [3]:
from domain.unit_of_work import SqlAlchemyUnitOfWork
from domain.models import Game, Platform, Tag

In [4]:
from domain.unit_of_work import SqlAlchemyUnitOfWork

uow = SqlAlchemyUnitOfWork()

print("PROVES DE REPOSITORI")

with uow:
    # 1. Test 'list' (Equivalent a get_all)
    tots = uow.games.list()
    print(f"Total de jocs a la DB: {len(tots)}")

    # 2. Cerca per títol (Query específica)
    titol_buscat = "Elden Ring"
    joc = uow.games.get_by_title(titol_buscat)
    if joc:
        print(f"Joc trobat: {joc.title} | Preu: {joc.price}€")

    # 3. Paginació
    # Demanem la pàgina 1 amb 2 elements
    p_jocs = uow.games.get_paginated(page=1, page_size=2)
    print(f"Paginació (Pàgina 1, mida 2): {[g.title for g in p_jocs]}")

PROVES DE REPOSITORI
Total de jocs a la DB: 2
Joc trobat: Elden Ring | Preu: 59.99€
Paginació (Pàgina 1, mida 2): ['Elden Ring', 'Mario Kart 8']


In [5]:
from domain.unit_of_work import SqlAlchemyUnitOfWork

uow = SqlAlchemyUnitOfWork()

# Intentem afegir el Tag al Joc dins del bloc 'with'
with uow:
    try:
        # Nota: Heu d'assegurar-vos que els IDs 1 i 2 existeixen a la DB
        uow.games.add_tag_to_game(game_id=1, tag_id=2)
        uow.commit() 
        print("Tag afegit correctament al joc!")
    except Exception as e:
        print(f"Error en l'operació: {e}")

print("\nTotes les proves finalitzades.")

Tag afegit correctament al joc!

Totes les proves finalitzades.


In [6]:
from domain.unit_of_work import SqlAlchemyUnitOfWork

uow = SqlAlchemyUnitOfWork()

print("TESTS DE TAG")

# Fem servir el bloc 'with' per a la Unit of Work
with uow:
    # 1. Test de Tags
    tots_els_tags = uow.tags.list()
    # Imprimim l'ID i el nom per demostrar que les dades són correctes
    tag_info = [f"{t.id}: {t.tag_name}" for t in tots_els_tags]
    print(f"Tags disponibles (ID: Nom): {tag_info}")

    # 2. Test de Ressenyes
    totes_les_reviews = uow.reviews.list()
    print(f"Total de ressenyes: {len(totes_les_reviews)}")

TESTS DE TAG
Tags disponibles (ID: Nom): ['6: Souls-like']
Total de ressenyes: 1


In [7]:
from domain.unit_of_work import SqlAlchemyUnitOfWork

uow = SqlAlchemyUnitOfWork()

with uow:
    print("TEST D'USUARIS:")
    tots_els_usuari = uow.users.list()
    for u in tots_els_usuari:
        print(f"Usuari: {u.username} | Email: {u.email}")

    print("\nDETALL DE RESSENYES:")
    totes_les_reviews = uow.reviews.list()
    for r in totes_les_reviews:
        # Aquí és on demostres les relacions entre taules
        print(f"{r.user.username} sobre '{r.game.title}': {r.score}/10 - {r.comment}")

TEST D'USUARIS:
Usuari: Marti | Email: marti@example.com

DETALL DE RESSENYES:
Marti sobre 'Elden Ring': 10/10 - Una obra mestra absoluta!


In [10]:
from domain.unit_of_work import SqlAlchemyUnitOfWork
from domain.models import Tag

# 1. Creem la unitat de treball
uow = SqlAlchemyUnitOfWork()

print("TESTS: GET(ID), UPDATE, DELETE ")

# Utilitzem el 'with' per gestionar la connexió
with uow:
    # --- 1. Test: Reading by ID ---
    joc_id_1 = uow.games.get(1)
    if joc_id_1:
        print(f"Test GET(ID): Joc trobat amb ID 1 -> {joc_id_1.title}")
    else:
        print("Test GET(ID): No s'ha trobat cap joc amb ID 1")

    # --- 2. Test: Updating ---
    if joc_id_1:
        preu_antic = joc_id_1.price
        # Canviem la propietat de l'objecte. SQLAlchemy ho detecta automàticament (Unit of Work "Tracking")
        joc_id_1.price = 99.99
        uow.commit() 
        
        # Tornem a consultar (el repositori ho llegirà de la sessió actual o la base de dades)
        joc_actualitzat = uow.games.get(1)
        print(f"Test UPDATE: Preu canviat de {preu_antic} a {joc_actualitzat.price}")

    # --- 3. Test: Deleting ---
    # Creem un tag temporal per esborrar-lo sense trencar les relacions dels jocs reals
    nou_tag = Tag(tag_name="Tag_Temporal_Per_Esborrar")
    uow.tags.add(nou_tag)
    uow.commit() 
    
    id_temporal = nou_tag.id
    print(f"Test DELETE: S'ha creat el tag temporal amb ID {id_temporal}")

    # L'esborrem usant el mètode delete del repositori
    uow.tags.delete(nou_tag)
    uow.commit() # Confirmem l'esborrat

    # Verifiquem que ha desaparegut
    verificacio = uow.tags.get(id_temporal)
    if verificacio is None:
        print(f"Test DELETE: Èxit! El tag amb ID {id_temporal} ja no existeix a la base de dades.")
    else:
        print("Test DELETE: Error, el tag encara existeix.")

TESTS: GET(ID), UPDATE, DELETE 
Test GET(ID): No s'ha trobat cap joc amb ID 1
Test DELETE: S'ha creat el tag temporal amb ID 9
Test DELETE: Èxit! El tag amb ID 9 ja no existeix a la base de dades.
